# General

For more informations, si the documentation *Documentary Strategy*.

# Import & Configs

In [1]:
from Bio import Entrez
import pandas as pd
from tqdm import tqdm
import json
import time

from pathlib import Path
import tomllib

In [2]:
def get_secrets():
    secrets_path = Path("../.secrets.toml")
    
    with open(secrets_path, "rb") as f:
        secrets = tomllib.load(f)
        
    return secrets

secrets = get_secrets()

In [3]:
Entrez.email = secrets["PUBMED_EMAIL"]

# PubMed extraction

PubMed query:

In [4]:
keywords = [
    "glioblastoma",
    "glioma",
    "meningioma",
    "pituitary tumor",
]

conditions = [
    "MRI",
    "treatment",
    "prognosis",
    "review[Publication Type]",
    "systematic review[Publication Type]",
    "meta-analysis[Publication Type]",
    "guideline[Publication Type]",
]

In [5]:
def construct_queries(keywords, conditions):
    queries = []
    for kw in keywords:
        queries.append(kw)
        for cond in conditions:
            queries.append(f'{kw} AND {cond}')
    return queries

In [6]:
QUERIES = construct_queries(keywords, conditions)
print(f"NB queries: {len(QUERIES)}\n")
QUERIES

NB queries: 32



['glioblastoma',
 'glioblastoma AND MRI',
 'glioblastoma AND treatment',
 'glioblastoma AND prognosis',
 'glioblastoma AND review[Publication Type]',
 'glioblastoma AND systematic review[Publication Type]',
 'glioblastoma AND meta-analysis[Publication Type]',
 'glioblastoma AND guideline[Publication Type]',
 'glioma',
 'glioma AND MRI',
 'glioma AND treatment',
 'glioma AND prognosis',
 'glioma AND review[Publication Type]',
 'glioma AND systematic review[Publication Type]',
 'glioma AND meta-analysis[Publication Type]',
 'glioma AND guideline[Publication Type]',
 'meningioma',
 'meningioma AND MRI',
 'meningioma AND treatment',
 'meningioma AND prognosis',
 'meningioma AND review[Publication Type]',
 'meningioma AND systematic review[Publication Type]',
 'meningioma AND meta-analysis[Publication Type]',
 'meningioma AND guideline[Publication Type]',
 'pituitary tumor',
 'pituitary tumor AND MRI',
 'pituitary tumor AND treatment',
 'pituitary tumor AND prognosis',
 'pituitary tumor AND

Get the pubmed id :

In [7]:
query = "glioblastoma AND MRI"

handle = Entrez.esearch(
    db="pubmed",
    term=query,
    retmax=200
)

record = Entrez.read(handle)

len(record["IdList"])

200

In [8]:
for query in QUERIES:

    print(f"\nSTART: {query}")

    print("  esearch...")
    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=200
    )

    print("  read...")
    record = Entrez.read(handle)

    print("  extract...")
    pmids = record["IdList"]

    print(
        f"  DONE -> {len(pmids)}"
    )

    time.sleep(1)


START: glioblastoma
  esearch...
  read...
  extract...
  DONE -> 200

START: glioblastoma AND MRI
  esearch...
  read...
  extract...
  DONE -> 200

START: glioblastoma AND treatment
  esearch...
  read...
  extract...
  DONE -> 200

START: glioblastoma AND prognosis
  esearch...
  read...
  extract...
  DONE -> 200

START: glioblastoma AND review[Publication Type]
  esearch...
  read...
  extract...
  DONE -> 200

START: glioblastoma AND systematic review[Publication Type]
  esearch...
  read...
  extract...
  DONE -> 200

START: glioblastoma AND meta-analysis[Publication Type]
  esearch...
  read...
  extract...
  DONE -> 200

START: glioblastoma AND guideline[Publication Type]
  esearch...
  read...
  extract...
  DONE -> 34

START: glioma
  esearch...
  read...
  extract...
  DONE -> 200

START: glioma AND MRI
  esearch...
  read...
  extract...
  DONE -> 200

START: glioma AND treatment
  esearch...
  read...
  extract...
  DONE -> 200

START: glioma AND prognosis
  esearch...
 

In [9]:
qsdqz

NameError: name 'qsdqz' is not defined

In [ ]:
pmids = []

for query in QUERIES:

    print(f"Searching: {query}")

    with Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=200
    ) as handle:

        record = Entrez.read(handle)

    pmids_query = record["IdList"]

    print(f"Found: {len(pmids)}")

    pmids.update(pmids_query)

    time.sleep(1)

In [11]:
pmids = []

for query in QUERIES:

    try:
        handle = Entrez.esearch(
            db="pubmed",
            term=query,
            retmax=200
        )

        record = Entrez.read(handle)

        pmids_query = record["IdList"]

        print(
            query,
            len(pmids_query)
        )

        for pmid in pmids_query:
            pmids.append({
                "query": query,
                "pmid": pmid
            })

    except Exception as e:
        print(
            f"ERROR on {query}: {e}"
        )

    time.sleep(1)

print(
    f"Total results: {len(pmids)}"
)

glioblastoma 200
glioblastoma AND MRI 200
glioblastoma AND treatment 200
glioblastoma AND prognosis 200
glioblastoma AND review[Publication Type] 200
glioblastoma AND systematic review[Publication Type] 200
glioblastoma AND meta-analysis[Publication Type] 200
glioblastoma AND guideline[Publication Type] 34
glioma 200
glioma AND MRI 200
glioma AND treatment 200
glioma AND prognosis 200
glioma AND review[Publication Type] 200
glioma AND systematic review[Publication Type] 200
glioma AND meta-analysis[Publication Type] 200
glioma AND guideline[Publication Type] 99
meningioma 200
meningioma AND MRI 200
meningioma AND treatment 200
meningioma AND prognosis 200
meningioma AND review[Publication Type] 200
meningioma AND systematic review[Publication Type] 200
meningioma AND meta-analysis[Publication Type] 200
meningioma AND guideline[Publication Type] 20
pituitary tumor 200
pituitary tumor AND MRI 200
pituitary tumor AND treatment 200
pituitary tumor AND prognosis 200
pituitary tumor AND revi

Download abstracts:

In [13]:
papers = []

for item in tqdm(pmids):

    try:
        pmid = item["pmid"]

        handle = Entrez.efetch(
            db="pubmed",
            id=pmid,
            rettype="medline",
            retmode="text"
        )

        text = handle.read()
        handle.close()

        papers.append({
            "query": item["query"],
            "pmid": pmid,
            "raw_text": text
        })

        time.sleep(0.3)

    except Exception as e:
        print(f"Error with {pmid}: {e}")

 74%|███████▍  | 4308/5801 [1:22:00<3:53:29,  9.38s/it]

Error with 29534265: IncompleteRead(4089 bytes read)


100%|██████████| 5801/5801 [1:40:14<00:00,  1.04s/it]  


In [14]:
len(papers)

5800

Save data as JSON Line:

In [15]:
output_path = "../data/raw/pubmed_abstracts.jsonl"

with open(output_path, "w") as f:

    for paper in papers:
        f.write(json.dumps(paper) + "\n")

Dataset loaded and saved.